In [ ]:
# use spark session if needed
from kpi_distribution_utils import analyze_kpi

from genpm.utils.consts import SHARED_DIR_PATH
from genpm.utils.utils import SparkDataManager

cfg = SPARK_CONFIGS["HALF_SAFE"]

spark = (
    SparkSession.builder.appName("GenPM-fe")
    .config(map=cfg)
    .config("spark.log.level", "ERROR")
    .getOrCreate()
)

EDA_DATA_PATH = SHARED_DIR_PATH / "eda_data"
raw_pm_path = EDA_DATA_PATH / "raw_pm_data"
pm_kpi_pivot = EDA_DATA_PATH / "pm_data_pivot"
sample_path = EDA_DATA_PATH / "sample"
pm_stats_path = EDA_DATA_PATH / "stats"
pm_agg_path = EDA_DATA_PATH / "agg"
pm_metadata = EDA_DATA_PATH / "pm_metadata"

PREPROCESSED_DATASET_PATH = SHARED_DIR_PATH / "preprocessed_dataset"
long_path = PREPROCESSED_DATASET_PATH / "pm_data_long"
wide_path = PREPROCESSED_DATASET_PATH / "pm_data_wide"

FINAL_PREPROCESSED_DATASET_PATH = PREPROCESSED_DATASET_PATH / "final"
final_kpi_defs_path = FINAL_PREPROCESSED_DATASET_PATH / "kpi_definitions"
final_pm_data_dataset_path = FINAL_PREPROCESSED_DATASET_PATH / "pm_data_dataset_preprocessed"
pm_df_long_idx_win_path = FINAL_PREPROCESSED_DATASET_PATH / "pm_df_long_indexed_winds"
final_simple_reports_path = FINAL_PREPROCESSED_DATASET_PATH / "simple_reports"

PREPROCESSED_DATASET_PATH = SHARED_DIR_PATH / "preprocessed_dataset"

In [ ]:
df_long_idx = spark.read.parquet(str(pm_df_long_idx_win_path))
df_pm_long = spark.read.parquet(str(long_path))

## Some thougths for KPI distributions:
- kpis have very different distributions, with occuring regime shifts and for some kpis, frequent periods with NO DATA or long-term periods with NO DATA
- kpis might be fully constant
- there are kpis with very little coverage, and big spikes from time to time, what is the source?

## Solutions
- kpis with different distributions shouldnt be naively standardized using z-score globally, but they should be standardized using robust standarization methods, combined with transformations fe. log1p, Box-Cox, Yeo-Johnson (exp. distributions). Robust methods should be prepared for regime change, but still include them after standardization.
- FE: Rolling window MinMax Scaler, rolling MAD or IQR

- No data long periods - periods where little bits of data are available at the start of a KPI, but then there is a long period of nulls, should be dropped and new starting time point should be set, where the data is truly (kinda) continous with minimal imputing

- Maybe we should assign NULL periods to a binary Dataframe? To model NULL distributions aswell?

- constant kpis should be flagged as consts, and returned as they are
- very little changing kpis, should be modeled as is, as they do sometimes have big spikes, and appear more like exp. distribution kpis

In [ ]:
# for kpi in kpis_list[:100]:
#     analyze_kpi(pm_df, kpi_id=kpi)

In [ ]:
kpis_list = [row["kpi_id"] for row in df_long_idx.select("kpi_id").distinct().collect()]

In [ ]:
for kp in kpi_list_final[:20]:
    analyze_kpi(pm_df, kpi_id=kp)

In [ ]:
kpis_list = [row["kpi_id"] for row in df_long_idx.select("kpi_id").distinct().collect()]